In [ ]:
import vr1
from vr1.core import FuelAssembly, Lattice
from vr1.settings import VR1Settings
from vr1.writer import WriterOpenMC
from vr1.plots import test_plots
from vr1.materials import VR1Materials
from vr1.VR1facility import Facility
import vr1.lattice_units as vlu
import openmc
from vr1.core import core_designs
import vr1.utils

In [ ]:
openmc.Materials.cross_sections = "<path_to_xml>" #must use ENDF VIII.0 for C12
mats = VR1Materials()
mats_object = mats.get_materials() #generates materials obj
mats_object.export_to_xml()

settings = openmc.Settings()
settings.export_to_xml()

# Lattice Building Tutorial

We now know how to make *LatticeUnitVR1* objects on their own, so let's put them into a lattice! 
To do this there's one more class we need to discuss: VR1Facility. 
This class essentially creates the geometry of the entire reactor without the lattice itself. 
Essentially this means it's the reactor vessel surrounded with concrete and filled with water. 
Now that you know how, why don't you plot it?

In [ ]:
facility = Facility()

Now we want to create a lattice to put into the facility. 
This is accomplished by representing the the reactor's gridplate as a list of lists in python where each element is a lattice unit and each nested list is a horizontal row. 
The easiest way to explain this is through example. 
Below is an example of an 8x8 lattice that contains only water. 

In [ ]:
water_lattice = [['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w'],
                 ['w','w','w','w','w','w','w','w']]

As you can tell, the code for a lattice unit of only water is "*w*". 
A table of the lattice codes is provided below. 
| Lattice Unit | Code |
| --- | --- |
| 4-plate Fuel Assembly | 4 |
| 6-plate Fuel Assembly | 6 |
| 8-plate Fuel Assembly | 8 |
| 4-plate Fuel Assmebly with Control Rod Fully Inserted | X4 |
| 4-plate Fuel Assmebly with Control Rod Fully Removed | O4 |
| 6-plate Fuel Assmebly with Control Rod Fully Inserted | X |
| 6-plate Fuel Assmebly with Control Rod Fully Inserted | X |
| 12mm VertChannel  |  v12  |
| 25mm VertChannel  |  v25  |
| 30mm VertChannel  |  v30  |
| 56mm VertChannel  |  v56  |
| Dummy Fuel Assembly | d |
| Graphite Reflector | G | 
| Beryllium Reflector | B| 
| Water | w | 
| N-plate Fuel Assembly with Control Rod Removed Xcm | *N*_*X* |
| N-plate Fuel Assembly with Ymm VertChannel inserted| v*Y*_*N* |



Now that you've got the codes in front of you, make a lattice input that describes a system with a row of three 8-plate fuel assemblies in a row.

In [ ]:
lattice_input = list

Let's look at your lattice. To insert this into the geometry, you use the *Lattice* object which has an argument called "*lattice_str*". This argument expects a list despite its name. It's a list of lists... of strings. So it's.. 

<br> 

After you've initialized the *Lattice* object, you just need to assign it to be the Facility's lattice with the argument *lattice* in the *build* method and you're done! There is no *build* call necessary on the Lattice class.

In [ ]:
lattice_obj = Lattice(materials=mats, lattice_str = lattice_input)

facility = Facility(materials=mats)
vr1_universe = facility.build(lattice=lattice_obj)

geometry = openmc.Geometry(root=vr1_universe)
geometry.export_to_xml()


import vr1.utils
vr1.utils.plot_vr1()

# Building a Real Core

Build this lattice and plot it. For absorption rods, you can implement them at whatever insertion amount you want. Do not worry about the green circles, the yellow circles, or the radial channel. 

<div>
<img src="figs/C1-B_configuration.png" width="500"/>
</div>

A table is provided to explain the meaning of the units.

<div>
<img src="figs/6plate.png" width="50"/>
</div>

6-plate assembly

<div>
<img src="figs/8plate.png" width="50"/>
</div>
8-plate assembly

<div>
<img src="figs/controlrod.png" width="50"/>
</div>
absorption rod, the letters do not matter for your model

<div>
<img src="figs/vertchannel.png" width="50"/>
</div>
vertical channel, feel free to estmiate the diameter

<div>
<img src="figs/dummy.png" width="50"/>
</div>
dummy fuel element

In [ ]:
lattice_input = list

lattice_obj = Lattice(materials=mats, lattice_str = lattice_input)

facility = Facility(materials=mats)
vr1_universe = facility.build(lattice=lattice_obj)

geometry = openmc.Geometry(root=vr1_universe)
geometry.export_to_xml()


import vr1.utils
vr1.utils.plot_vr1()

# Checking k-eff

To check the k-eff of our core, we'll have to create some settings. This has been done before you below. 
Make sure you understand the settings. Particularly, notice how the source area is defined. 
Do you understand what is happening there? 
The neutrons are being restricted to be born inside the lattice rather than in the surrounding water of the pool. 

In [ ]:
settings = openmc.Settings()
settings.run_mode = 'eigenvalue'
settings.temperature = {'method':'interpolation','range':(293.15,923.15)} #unsure
settings.batches = 30
settings.inactive = 15 
settings.particles = 5000
settings.photon_transport = False
source_area = openmc.stats.Box(lattice_obj.source_lower_left,lattice_obj.source_upper_right)
settings.source = openmc.Source(space=source_area,constraints={'fissionable': True})
settings.export_to_xml()

With these settings created, all that needs to be done is to call *openmc.run()*! 

In [ ]:
openmc.run()

# GUI Tutorial

We have also added the ability to build a core using a GUI. This can be fun and more interactive, but for coding purposes it is easier to just do it by hand. Regardless, we'll explore the GUI here. The GUI will allow you to type in your code design to pre-generated squares for each element location. The two buttons "Save Configuration" and "Export Configuration" do slightly different things. 

"Save Configuration" will return the lattice in that you've created in list form to whatever variable you've assigned the call to. For example, to generate a lattice via GUI and then continue your code, you would write

gui_object = vr1.gui.VR1LatticeBuilder()<br>
lattice_gui = gui_object.run()

and then lattice_gui would be equivalent to lattice_str from earlier in the exercise.

"Export Configuration" will save the lattice to a file with extension ".lat" that contains the string rather than assigning it to a variable.

Try it below! Make a lattice in the GUI and plot it to make sure it looks how you wanted.

In [ ]:
import vr1.gui



# Core Building Challenge

Experiment with lattices to create a high k $_{eff}$. The design with the highest k $_{eff}$ wins a prize!

In [ ]:
#design core here

In [ ]:
settings = openmc.Settings()
settings.run_mode = 'eigenvalue'
settings.temperature = {'method':'interpolation','range':(293.15,923.15)} #unsure
settings.batches = 30
settings.inactive = 15 
settings.particles = 10000
settings.photon_transport = False
source_area = openmc.stats.Box(lattice_obj.source_lower_left,lattice_obj.source_upper_right)
settings.source = openmc.Source(space=source_area,constraints={'fissionable': True})
settings.export_to_xml()

openmc.run()